# Does order book imbalance predict the next move?

The book channel gives a factor at every frame. Whether that factor carries
information is a separate question, and this notebook asks it.

**Read the limitations first, not last.** Everything below is in-sample, on one
symbol, on one venue, gross of fees and gross of the spread that would have to
be crossed to act on any of it. A touch-level signal is exactly the kind that
the spread eats. Nothing here is a strategy, and nothing here has been tested
out of sample.

The arithmetic lives in `l2tca.research.evaluate` and is unit-tested. This
notebook is a presentation layer: if you find yourself writing analysis here,
it belongs in the module instead, where it can be tested.

## Input

Produced by replaying a capture through the book:

```bash
l2tca signals data/raw/<capture>.jsonl.gz --out data/parquet
```

That writes the `snapshot` and `signal` tables. Every row carries the checksum
verdict of the frame that produced it, so a book the exchange disagreed with is
visible rather than silently mixed in.

In [ ]:
import polars as pl

from l2tca.io.reader import read_table
from l2tca.research import (
    bucket_summary,
    forward_return_bps,
    information_coefficient,
    signals_wide,
)

ROOT = "data/parquet"
MS = 1_000_000

snapshots = read_table(ROOT, "snapshot")
wide = signals_wide(read_table(ROOT, "signal"))

span_s = (wide["recv_ns"].max() - wide["recv_ns"].min()) / 1e9
print(f"{wide.height:,} book states over {span_s / 60:.1f} minutes")
print(snapshots["checksum_ok"].value_counts())

### Is the sample long enough to ask the question?

A few minutes of one symbol cannot separate a property of the venue from a
property of the particular minute. Below roughly an hour, treat every number in
this notebook as machinery working rather than as a finding.

In [ ]:
if span_s < 3600:
    print(f"WARNING: {span_s / 60:.1f} minutes. Too short to conclude anything.")
    print("Record a longer session:  l2tca record --trades --duration 21600 --compress")

## What the book looked like

Before asking what a factor predicts, look at its distribution. A factor pinned
near one end of its range has little left to explain anything with.

In [ ]:
wide.select(["imbalance_1", "imbalance_5", "quoted_spread_bps"]).describe()

## Information coefficient across horizons

Spearman rank correlation between the factor and the subsequent mid move.
Rank rather than Pearson: both series are heavy-tailed, and the question is
whether the *ordering* carries information, not whether the outliers line up.

A signal that is real should not change sign as the horizon moves. One that
does is telling you the sample is too small.

In [ ]:
horizons_ms = [50, 100, 250, 500, 1000, 2000, 5000, 10_000]
factors = [c for c in wide.columns if c.startswith("imbalance")]

rows = []
for h in horizons_ms:
    frame = forward_return_bps(wide, h * MS)
    usable = frame["forward_bps"].drop_nulls().len()
    for factor in factors:
        rows.append(
            {
                "horizon_ms": h,
                "factor": factor,
                "n": usable,
                "ic": information_coefficient(frame, factor),
            }
        )

ic_table = pl.DataFrame(rows)
ic_table.pivot(on="factor", index=["horizon_ms", "n"], values="ic")

## Bucketed forward return

The IC is one number for the whole sample and hides everything. This is the
shape: sort by the factor, cut into deciles, and report the mean forward move
in each.

`forward_stderr` is the column to read first. A tidy monotone `forward_mean`
means nothing if the standard error of each bucket is larger than the distance
between them.

In [ ]:
HORIZON_MS = 1000
FACTOR = "imbalance_1"

frame = forward_return_bps(wide, HORIZON_MS * MS)
table = bucket_summary(frame, FACTOR, buckets=10)
table

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(
    table["bucket"], table["forward_mean"], yerr=table["forward_stderr"],
    marker="o", capsize=3,
)
ax.axhline(0, linewidth=0.8, color="grey")
ax.set_xlabel(f"{FACTOR} decile  (1 = most ask-heavy)")
ax.set_ylabel(f"mean mid move over {HORIZON_MS} ms  (bps)")
ax.set_title("Forward return by imbalance decile")
fig.tight_layout()

## Where the spread sits

The last column of the argument, and the one that usually ends it. Whatever the
buckets show, acting on the signal means crossing the spread. If the spread is
wider than the spread between the extreme buckets, the signal is not tradeable
however clean its shape.

In [ ]:
spread = wide["quoted_spread_bps"]
edge = table["forward_mean"].max() - table["forward_mean"].min()
print(f"median quoted spread : {spread.median():.4f} bps")
print(f"top-to-bottom decile spread in forward return : {edge:.4f} bps")
print(f"ratio : {edge / spread.median():.2f}x   (below 1 means the spread eats it)")

## What would make this stronger

- **More data.** Hours across sessions, not minutes. The first thing to fix.
- **Out of sample.** Fit the buckets on one session, check them on the next.
- **Trade sign.** The `trade` table carries the aggressor's side directly, so
  signed order flow can be built without a Lee-Ready style inference. That is
  the natural second factor, and it is a different kind of information from a
  resting-quantity imbalance.
- **A cost model.** The spread comparison above is the crudest possible version.